# 03b — Silver Layer: Fact Table — silver_market_metrics

In [0]:
# ══════════════════════════════════════════════════════════════════════════
# SILVER LAYER — FACT TABLE : silver_market_metrics  (market fact)
# Notebook : 03b_silver_fact_market.py
#
# Data model : star schema
#   Grain     : 1 row per (coin_id, ingestion_date)
#   Dim joins : dim_coin  (id)
#               dim_date  (ingestion_date → date_id)
#
# Output table silver_market_metrics keeps the EXACT same column set
# as before — the gold layer (05_gold_trader_view) needs zero changes.
#
# What changed vs old 03_transform_market:
#   • Coin metadata (rank, segment, supply) now live in dim_coin —
#     fact table joins them instead of carrying raw copies.
#   • ingestion_date is joined to dim_date to add date_id, year_month
#     and year_quarter for richer BI slicing.
#   • Quarantine, DQ, anomaly, feature-engineering logic is unchanged.
#   • Silver MERGE key is still (id, ingestion_date).
# ══════════════════════════════════════════════════════════════════════════

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType, TimestampType
from datetime import datetime, timezone

DB_NAME               = 'crypto_db'
BRONZE_MARKET         = f'{DB_NAME}.bronze_market_data'
DIM_COIN              = f'{DB_NAME}.dim_coin'
DIM_DATE              = f'{DB_NAME}.dim_date'
SILVER_MARKET         = f'{DB_NAME}.silver_market_metrics'
SILVER_MARKET_HISTORY = f'{DB_NAME}.silver_market_history'
QUARANTINE_TABLE      = f'{DB_NAME}.silver_market_quarantine'
DQ_LOG_TABLE          = f'{DB_NAME}.silver_data_quality_log'

def _now_utc():
    return datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')

def _today_utc():
    return datetime.now(timezone.utc).strftime('%Y-%m-%d')

print(f'Config loaded — target: {SILVER_MARKET}')

In [0]:
# ── 1. Load Bronze & Basic Preprocessing ─────────────────────────────────

df = spark.table(BRONZE_MARKET)
df = df.toDF(*[c.lower().strip().replace(' ', '_') for c in df.columns])

DROP_COLS = ['image', 'roi', 'is_fallback', 'archive_run_id',
             'price_change_percentage_24h_in_currency']
df = df.drop(*[c for c in DROP_COLS if c in df.columns])

NUMERIC_COLS = [
    'current_price', 'market_cap', 'total_volume', 'high_24h', 'low_24h',
    'price_change_24h', 'price_change_percentage_24h',
    'market_cap_change_24h', 'market_cap_change_percentage_24h',
    'circulating_supply', 'total_supply', 'max_supply',
    'ath', 'ath_change_percentage', 'atl', 'atl_change_percentage',
    'fully_diluted_valuation'
]
for c in NUMERIC_COLS:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast(DoubleType()))

if 'market_cap_rank' in df.columns:
    df = df.withColumn('market_cap_rank', F.col('market_cap_rank').cast(IntegerType()))

if 'ingestion_timestamp' in df.columns:
    df = df.withColumn('ingestion_timestamp',
                       F.col('ingestion_timestamp').cast(TimestampType()))

for col_name, dtype in df.dtypes:
    if dtype == 'string':
        df = df.withColumn(col_name, F.trim(F.col(col_name)))

df = (df
    .withColumn('processing_date',  F.col('ingestion_date').cast('date'))
    .withColumn('processing_week',  F.weekofyear(F.col('ingestion_date').cast('date')))
    .withColumn('processing_month', F.month(F.col('ingestion_date').cast('date')))
    .withColumn('processing_year',  F.year(F.col('ingestion_date').cast('date')))
)

# Deduplicate (latest per coin per day)
w_dedup = Window.partitionBy('id', 'ingestion_date') \
                .orderBy(F.col('ingestion_timestamp').desc())
df = (df.withColumn('_rn', F.row_number().over(w_dedup))
        .filter(F.col('_rn') == 1)
        .drop('_rn'))

total_raw = df.count()
print(f'Bronze loaded after dedup: {total_raw} rows')

In [0]:
# ── 2. Join dim_coin — enrich with master coin attributes ─────────────────
# We only pull the dim keys + segment here; gold still gets market_cap_rank
# and market_segment from the fact row (carried through from dim join).

dim_coin = spark.table(DIM_COIN).select(
    'id',
    'market_segment',           # overwrite if bronze differs (SCD-1)
    'supply_utilisation_pct',
    'ath_distance_bucket',
    'is_scarce',
)

df = df.join(dim_coin, on='id', how='left')
print(f'After dim_coin join: {df.count()} rows (should stay same)')

In [0]:
# ── 3. Join dim_date — add calendar surrogate + BI-friendly labels ────────

dim_date = spark.table(DIM_DATE).select(
    F.col('date_actual').alias('ingestion_date'),
    'date_id',
    'year_month',
    'year_quarter',
    'year_week',
    'day_name',
    'is_weekend',
    'trading_session',       # not present in dim_date — added below
)

# dim_date does not have trading_session (that is in dim_date_hour)
# so we drop it from the select and add a simple crypto_always_open flag
dim_date = spark.table(DIM_DATE).select(
    F.col('date_actual').alias('ingestion_date'),
    'date_id',
    'year_month',
    'year_quarter',
    'year_week',
    'day_name',
    'is_weekend',
)

df = df.join(dim_date, on='ingestion_date', how='left')
print(f'After dim_date join: {df.count()} rows')

In [0]:
# ── 4. Data Quality Validation & Quarantine ───────────────────────────────

invalid_condition = (
    F.col('id').isNull() |
    F.col('current_price').isNull() | (F.col('current_price') <= 0) |
    F.col('market_cap').isNull()    | (F.col('market_cap') <= 0) |
    F.col('total_volume').isNull()  | (F.col('total_volume') < 0)
)

df_quarantine = df.filter(invalid_condition)
df_clean      = df.filter(~invalid_condition)

df_quarantine = df_quarantine.withColumn(
    'quarantine_reason',
    F.when(F.col('id').isNull(),                'missing_id')
     .when(F.col('current_price').isNull(),     'missing_price')
     .when(F.col('current_price') <= 0,         'invalid_price')
     .when(F.col('market_cap').isNull(),        'missing_market_cap')
     .when(F.col('market_cap') <= 0,            'invalid_market_cap')
     .when(F.col('total_volume').isNull(),      'missing_volume')
     .when(F.col('total_volume') < 0,           'invalid_volume')
).withColumn('quarantine_timestamp', F.lit(_now_utc()))

df_clean = df_clean.fillna({
    'price_change_24h': 0.0,
    'price_change_percentage_24h': 0.0,
    'market_cap_change_24h': 0.0,
    'market_cap_change_percentage_24h': 0.0,
    'high_24h': 0.0,
    'low_24h': 0.0,
})

valid_count      = df_clean.count()
quarantine_count = df_quarantine.count()
print(f'Valid rows: {valid_count}')
print(f'Quarantined rows: {quarantine_count}')

In [0]:
# ── 5. Anomaly Detection (dynamic thresholds — unchanged) ─────────────────

price_lower, price_upper = df_clean.approxQuantile(
    'price_change_percentage_24h', [0.01, 0.99], 0.01
)

df_clean = df_clean.withColumn(
    'volume_mc_ratio',
    F.when(F.col('market_cap') > 0,
           F.col('total_volume') / F.col('market_cap'))
)

volume_upper = df_clean.approxQuantile('volume_mc_ratio', [0.99], 0.01)[0]

df_clean = df_clean.withColumn(
    'price_change_pct_24h_capped',
    F.when(F.col('price_change_percentage_24h') > price_upper, price_upper)
     .when(F.col('price_change_percentage_24h') < price_lower, price_lower)
     .otherwise(F.col('price_change_percentage_24h'))
)

df_clean = (df_clean
    .withColumn('price_anomaly',
        (F.col('price_change_percentage_24h') < price_lower) |
        (F.col('price_change_percentage_24h') > price_upper))
    .withColumn('volume_anomaly',
        F.col('volume_mc_ratio') > volume_upper)
    .withColumn('anomaly_flag',
        F.col('price_anomaly') | F.col('volume_anomaly'))
    .withColumn('anomaly_reason',
        F.concat_ws(',',
            F.when(F.col('price_anomaly'),  F.lit('PRICE_OUTLIER')),
            F.when(F.col('volume_anomaly'), F.lit('VOLUME_SPIKE'))
        ))
)

anomaly_count = df_clean.filter(F.col('anomaly_flag')).count()
print(f'Anomalies flagged: {anomaly_count}')

In [0]:
# ── 6. Feature Engineering (unchanged — gold depends on these) ────────────

w_coin    = Window.partitionBy('id').orderBy('ingestion_timestamp')
w_coin_3d = w_coin.rowsBetween(-2, 0)

df_clean = (df_clean
    .withColumn('prev_price',  F.lag('current_price').over(w_coin))
    .withColumn('price_lag_2', F.lag('current_price', 2).over(w_coin))
    .withColumn('prev_rank',   F.lag('market_cap_rank').over(w_coin))
)

# 6a. Price movement
df_clean = (df_clean
    .withColumn('daily_return_pct',
        F.when(F.col('prev_price') > 0,
            ((F.col('current_price') - F.col('prev_price')) / F.col('prev_price')) * 100
        ).otherwise(F.col('price_change_pct_24h_capped')))
    .withColumn('price_3d_change_pct',
        F.when(F.col('price_lag_2') > 0,
            ((F.col('current_price') - F.col('price_lag_2')) / F.col('price_lag_2')) * 100))
    .withColumn('price_movement_metric',
        F.when(F.col('price_3d_change_pct').isNotNull(),
            (F.col('daily_return_pct') + F.col('price_3d_change_pct')) / 2
        ).otherwise(F.col('daily_return_pct')))
)

# 6b. Short-term trend
df_clean = (df_clean
    .withColumn('ma_3', F.avg('current_price').over(w_coin_3d))
    .withColumn('trend_strength_ratio',
        F.when(F.col('ma_3') > 0, F.col('current_price') / F.col('ma_3')))
    .withColumn('short_term_trend_signal',
        F.when(F.col('trend_strength_ratio') > 1, 'BULLISH')
         .when(F.col('trend_strength_ratio') < 1, 'BEARISH')
         .otherwise('NEUTRAL'))
)

# 6c. Price range volatility (3-day window)
df_clean = (df_clean
    .withColumn('price_3d_max', F.max('current_price').over(w_coin_3d))
    .withColumn('price_3d_min', F.min('current_price').over(w_coin_3d))
    .withColumn('price_range_volatility',
        F.when(F.col('price_3d_min') > 0,
            ((F.col('price_3d_max') - F.col('price_3d_min')) / F.col('price_3d_min')) * 100))
)

# 6d. Liquidity strength
df_clean = (df_clean
    .withColumn('liquidity_strength',
        F.when((F.col('market_cap') > 0) & F.col('total_volume').isNotNull(),
            F.col('total_volume') / F.col('market_cap')))
    .withColumn('liquidity_strength',      F.round('liquidity_strength', 4))
    .withColumn('liquidity_strength_pct',  F.round(F.col('liquidity_strength') * 100, 2))
)

# 6e. Rank movement
df_clean = df_clean.withColumn('rank_movement',
    F.when(F.col('prev_rank').isNotNull(),
        F.col('prev_rank') - F.col('market_cap_rank')))

# 6f. Data freshness
df_clean = df_clean.withColumn('data_freshness_hours',
    F.when(F.col('ingestion_timestamp').isNotNull(),
        F.round(
            (F.unix_timestamp(F.current_timestamp()) -
             F.unix_timestamp('ingestion_timestamp')) / 3600, 2)))

print('✅ Feature engineering complete')

In [0]:
# ── 7. Change Detection ───────────────────────────────────────────────────

change_threshold = df_clean.approxQuantile('daily_return_pct', [0.90], 0.01)[0]

df_clean = df_clean.withColumn('is_changed',
    F.when(F.col('daily_return_pct').isNull(), False)
     .when(F.abs(F.col('daily_return_pct')) > change_threshold, True)
     .otherwise(False))

print(f'✅ Change threshold (dynamic): {change_threshold}')

# Drop intermediate cols
df_clean = df_clean.drop('prev_price', 'price_3d_max', 'price_3d_min',
                          'price_anomaly', 'volume_anomaly', 'volume_mc_ratio')

# Null fill for derived cols
df_clean = df_clean.fillna({
    'price_3d_change_pct': 0.0,
    'price_range_volatility': 0.0,
    'rank_movement': 0
})

changed_count = df_clean.filter(F.col('is_changed')).count()
total_count   = df_clean.count()
print(f'✅ Final silver rows: {total_count} | Changed: {changed_count}')

In [0]:
# ── 8. Write silver_market_metrics (idempotent MERGE) ────────────────────
# Column selection mirrors the OLD silver table so gold layer is unaffected.
# New dim-join columns (date_id, year_month, year_quarter, year_week,
# day_name, is_weekend, supply_utilisation_pct, ath_distance_bucket,
# is_scarce) are ADDED — they are additive and do not break gold selects.

df_clean.createOrReplaceTempView('_silver_market_temp')

exists = spark.catalog.tableExists(SILVER_MARKET)
if not exists:
    (df_clean.write.format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .partitionBy('ingestion_date')
        .saveAsTable(SILVER_MARKET))
    print(f'Created {SILVER_MARKET}')
else:
    source_cols = set(df_clean.columns)
    target_cols = set(spark.table(SILVER_MARKET).columns)
    common_cols = list(source_cols & target_cols)
    set_clause  = ', '.join([f'tgt.{c} = src.{c}' for c in common_cols])
    ins_cols    = ', '.join(common_cols)
    ins_vals    = ', '.join([f'src.{c}' for c in common_cols])
    spark.sql(f"""
        MERGE INTO {SILVER_MARKET} tgt
        USING _silver_market_temp src
        ON tgt.id = src.id AND tgt.ingestion_date = src.ingestion_date
        WHEN MATCHED     THEN UPDATE SET {set_clause}
        WHEN NOT MATCHED THEN INSERT ({ins_cols}) VALUES ({ins_vals})
    """)
    print(f'Merged {SILVER_MARKET}')

# History append (full audit trail)
hist_exists = spark.catalog.tableExists(SILVER_MARKET_HISTORY)
(df_clean.write.format('delta')
    .mode('append' if hist_exists else 'overwrite')
    .option('mergeSchema', 'true')
    .partitionBy('ingestion_date')
    .saveAsTable(SILVER_MARKET_HISTORY))
print(f'History appended — {SILVER_MARKET_HISTORY}')

# Quarantine
(df_quarantine.write.format('delta')
    .mode('append')
    .saveAsTable(QUARANTINE_TABLE))
print(f'Quarantine updated: {quarantine_count} rows')

In [0]:
# ── 9. OPTIMIZE ───────────────────────────────────────────────────────────

try:
    spark.sql(f'OPTIMIZE {SILVER_MARKET} ZORDER BY (id, market_cap_rank)')
    print(f'OPTIMIZE done on {SILVER_MARKET}')
except Exception as e:
    print(f'OPTIMIZE skipped: {e}')

In [0]:
# ── 10. Verification ──────────────────────────────────────────────────────

print('\n====== DATA VERIFICATION ======')

print('\n--- Silver Market Preview (top 5) ---')
spark.table(SILVER_MARKET).select(
    'id', 'ingestion_date', 'date_id', 'year_month',
    'current_price', 'price_movement_metric', 'short_term_trend_signal',
    'price_range_volatility', 'liquidity_strength',
    'market_segment', 'anomaly_flag', 'is_changed',
    'supply_utilisation_pct', 'ath_distance_bucket'
).orderBy(F.col('market_cap_rank')).display(5, truncate=False)

print('\n--- Bitcoin Detail ---')
spark.table(SILVER_MARKET).filter(F.col('id') == 'bitcoin').select(
    'id', 'ingestion_date', 'year_quarter', 'current_price',
    'ma_3', 'daily_return_pct', 'price_range_volatility',
    'short_term_trend_signal', 'rank_movement', 'data_freshness_hours',
    'anomaly_flag', 'is_scarce', 'ath_distance_bucket'
).display(5, truncate=False)

print('\n--- Duplicate check ---')
spark.sql(f'''
    SELECT id, ingestion_date, COUNT(*) as cnt
    FROM {SILVER_MARKET}
    GROUP BY id, ingestion_date HAVING cnt > 1
''').display()

print('\n--- Market segment distribution ---')
spark.table(SILVER_MARKET).groupBy('market_segment').count() \
    .orderBy('count', ascending=False).display()

print('\n--- New dim-enriched fields sample ---')
spark.table(SILVER_MARKET).select(
    'id', 'ingestion_date', 'date_id', 'year_month', 'year_quarter',
    'year_week', 'day_name', 'is_weekend',
    'supply_utilisation_pct', 'is_scarce', 'ath_distance_bucket'
).limit(5).display(truncate=False)

print('\n✅ silver_market_metrics complete.')